# Music Energy / Stress Analysis

Downloads the first **N songs** from a public-domain YouTube playlist, then visualises
how the energy and 'stress' of each song evolve over time.

**Stress score** = weighted blend of three features:

| Feature | Weight | What it captures |
|---|---|---|
| RMS energy | 50 % | Loudness / power |
| Spectral centroid | 25 % | Brightness (more treble → higher) |
| Onset strength | 25 % | Rhythmic density (attacks per second) |

The smoothed curve lets you clearly see **intro → build-up → chorus → outro** transitions.

In [1]:
# Run once to install all dependencies into this kernel's Python
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'yt-dlp', 'librosa', 'soundfile', '-q'])
print('Dependencies ready.')

Dependencies ready.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.ndimage import uniform_filter1d
from pathlib import Path

print('Libraries loaded.')

Libraries loaded.


## Configuration

Adjust the constants below before running anything else.

In [ ]:
import re

PLAYLIST_URL = 'https://www.youtube.com/playlist?list=PLh5X0e7-mnI3lh-nb3fwZ8OcndGLfVdZX'
N_SONGS      = 10       # how many songs to grab from the playlist

# Folder is named after the playlist ID — swap the URL and you get a separate folder automatically
_playlist_id = re.search(r'[?&]list=([^&]+)', PLAYLIST_URL).group(1)
DOWNLOAD_DIR = Path('downloads') / _playlist_id

SR           = 22050    # sample rate used by librosa
HOP_LENGTH   = 512      # frame step  ≈ 23 ms
SMOOTH_SEC   = 5        # rolling-window size for the macro stress curve (seconds)

print(f'Playlist ID : {_playlist_id}')
print(f'Download dir: {DOWNLOAD_DIR}')

## Download

> **Run this cell only when you are ready to download.**  
> Songs are saved as MP3 files inside `downloads/<playlist-id>/`.  
> Re-running is safe — already-downloaded tracks are skipped automatically.

In [ ]:
import yt_dlp

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
_archive = str(DOWNLOAD_DIR / '.yt-dlp-archive')  # tracks downloaded IDs → skip on re-run

ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': str(DOWNLOAD_DIR / '%(playlist_index)02d_%(title)s.%(ext)s'),
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
        'preferredquality': '192',
    }],
    'playlistend': N_SONGS,
    'quiet': False,
    'ignoreerrors': True,
    'nooverwrites': True,
    'download_archive': _archive,
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([PLAYLIST_URL])

files = sorted(DOWNLOAD_DIR.glob('*.mp3'))
print(f'\n{len(files)} file(s) in {DOWNLOAD_DIR}:')
for f in files:
    print(' ', f.name)

## Analysis

1. Run **List files** to see what was downloaded.
2. Set `FILE_IDX` to the song you want to analyse.
3. Run the remaining cells in order.

In [ ]:
audio_files = sorted(DOWNLOAD_DIR.glob('*.mp3'))
if not audio_files:
    print(f'No MP3 files found in {DOWNLOAD_DIR} — run the Download cell first.')
else:
    for i, f in enumerate(audio_files):
        print(f'[{i}]  {f.name}')

In [6]:
FILE_IDX = 0  # ← change to pick a different song

audio_path = audio_files[FILE_IDX]
print(f'Loading: {audio_path.name}')

y, sr = librosa.load(str(audio_path), sr=SR, mono=True)
duration = len(y) / sr
print(f'Duration : {duration:.1f} s  ({duration / 60:.2f} min)')
print(f'Samples  : {len(y):,}')

Loading: 01_Spring In My Step - ANONYMOUS #music #pop.mp3
Duration : 119.2 s  (1.99 min)
Samples  : 2,627,585


### Feature extraction

Each feature is computed per *frame* (~23 ms), then the stress score is smoothed
over a `SMOOTH_SEC`-wide rolling window to expose the macro song structure.

In [ ]:
def compute_features(fpath, sr=SR, hop_length=HOP_LENGTH):
    """Return (rms, centroid, onset_env, times) — loads from .npz cache if available."""
    cache = fpath.with_suffix('.features.npz')
    if cache.exists():
        d = np.load(cache)
        return d['rms'], d['centroid'], d['onset_env'], d['times']
    y, _ = librosa.load(str(fpath), sr=sr, mono=True)
    rms       = librosa.feature.rms(y=y, hop_length=hop_length)[0]
    centroid  = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop_length)[0]
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length)
    times     = librosa.times_like(rms, sr=sr, hop_length=hop_length)
    np.savez(cache, rms=rms, centroid=centroid, onset_env=onset_env, times=times)
    return rms, centroid, onset_env, times

In [ ]:
def norm01(x):
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo + 1e-8)

_cached = audio_path.with_suffix('.features.npz').exists()
rms, centroid, onset_env, times = compute_features(audio_path)
print(f'Features {"loaded from cache" if _cached else "computed and cached"}  |  {len(times):,} frames')

# Weighted stress score (raw, per-frame)
stress_raw = (
    0.50 * norm01(rms) +
    0.25 * norm01(centroid) +
    0.25 * norm01(onset_env)
)

# Smooth → macro song structure
smooth_frames = int(SMOOTH_SEC * SR / HOP_LENGTH)
stress_smooth = uniform_filter1d(stress_raw, size=smooth_frames)
stress_smooth = norm01(stress_smooth)

print(f'Stress score ready  |  smoothing window = {smooth_frames} frames ({SMOOTH_SEC} s)')

### Visualisation

Three panels:
- **Top** – stress time series (raw + smoothed)
- **Middle** – individual feature breakdown
- **Bottom** – mel spectrogram (frequency content over time)

In [ ]:
def fmt_time(x, _):
    m, s = divmod(int(x), 60)
    return f'{m}:{s:02d}'

fig, axes = plt.subplots(
    3, 1, figsize=(15, 11),
    gridspec_kw={'height_ratios': [3, 1.5, 2]}
)
fig.suptitle(
    f'Music Energy / Stress Analysis\n{audio_path.stem}',
    fontsize=13, fontweight='bold'
)

# ── Panel 1: stress ─────────────────────────────────────────────────────────
ax1 = axes[0]
ax1.fill_between(times, stress_raw,    alpha=0.12, color='steelblue')
ax1.fill_between(times, stress_smooth, alpha=0.25, color='crimson')
ax1.plot(times, stress_raw,    color='steelblue', linewidth=0.6,
         alpha=0.5,  label='Stress — raw (per frame ≈ 23 ms)')
ax1.plot(times, stress_smooth, color='crimson',   linewidth=2.5,
         label=f'Stress — smoothed ({SMOOTH_SEC} s window)')
ax1.set_ylabel('Stress score  (0 – 1)')
ax1.set_ylim(-0.05, 1.05)
ax1.legend(loc='upper right')
ax1.set_title('Song Energy / Stress Over Time')
ax1.xaxis.set_major_formatter(ticker.FuncFormatter(fmt_time))
ax1.set_xlabel('Time (mm:ss)')
ax1.grid(True, alpha=0.3)

# ── Panel 2: feature breakdown ───────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(times, norm01(rms),       color='#e67e22', linewidth=1.2,
         label='RMS energy  (loudness)')
ax2.plot(times, norm01(centroid),  color='#27ae60', linewidth=1.2,
         label='Spectral centroid  (brightness)')
ax2.plot(times, norm01(onset_env), color='#8e44ad', linewidth=1.2,
         label='Onset strength  (rhythmic density)')
ax2.set_ylabel('Normalised value')
ax2.set_ylim(-0.05, 1.05)
ax2.legend(loc='upper right', fontsize=8)
ax2.xaxis.set_major_formatter(ticker.FuncFormatter(fmt_time))
ax2.set_xlabel('Time (mm:ss)')
ax2.grid(True, alpha=0.3)
ax2.set_title('Feature Breakdown')

# ── Panel 3: mel spectrogram ─────────────────────────────────────────────────
ax3 = axes[2]
S    = librosa.feature.melspectrogram(y=y, sr=SR, hop_length=HOP_LENGTH, n_mels=128)
S_dB = librosa.power_to_db(S, ref=np.max)
img  = librosa.display.specshow(
    S_dB, sr=SR, hop_length=HOP_LENGTH,
    x_axis='time', y_axis='mel', ax=ax3, cmap='magma'
)
ax3.set_title('Mel Spectrogram  (frequency content over time)')
fig.colorbar(img, ax=ax3, format='%+2.0f dB')

plt.tight_layout()
out_png = DOWNLOAD_DIR / (audio_path.stem + '_stress.png')
plt.savefig(str(out_png), dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved → {out_png}')

## Batch: stress curves for all songs

Overlay the smoothed stress curve of every downloaded song in a single plot
so you can compare their energy arcs at a glance.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))

cmap = plt.get_cmap('tab10')
for idx, fpath in enumerate(audio_files):
    _cached = fpath.with_suffix('.features.npz').exists()
    _rms, _cent, _ons, _t = compute_features(fpath)
    _s  = 0.50 * norm01(_rms) + 0.25 * norm01(_cent) + 0.25 * norm01(_ons)
    _sf = int(SMOOTH_SEC * SR / HOP_LENGTH)
    _s  = norm01(uniform_filter1d(_s, size=_sf))
    ax.plot(_t, _s, linewidth=1.5, alpha=0.8,
            color=cmap(idx % 10), label=fpath.stem[:40])
    print(f'[{idx}] {"(cache)" if _cached else "(computed)"}  {fpath.name}')

ax.set_title('Stress Curves — All Songs', fontsize=13, fontweight='bold')
ax.set_ylabel('Stress score  (0 – 1)')
ax.set_xlabel('Time (mm:ss)')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(fmt_time))
ax.legend(fontsize=7, loc='upper right', ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
out_batch = DOWNLOAD_DIR / 'all_songs_stress.png'
plt.savefig(str(out_batch), dpi=150, bbox_inches='tight')
plt.show()
print(f'Batch plot saved → {out_batch}')